<a href="https://colab.research.google.com/github/Rehan07i/Ai_Quize/blob/main/pothole_detection_%26mapping_system_for_rainy_conditions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 — Install YOLO
!pip -q install ultralytics

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("atulyakumar98/pothole-detection-dataset")

print("Path to dataset files:", path)


100%|██████████| 194M/194M [00:01<00:00, 140MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/atulyakumar98/pothole-detection-dataset/versions/4


In [3]:
import os

for root, dirs, files in os.walk(path):
    level = root.replace(path, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files[:10]:
        print(f"{indent}  {file}")

4/
  normal/
    210.jpg
    78.jpg
    89.jpg
    169.jpg
    256.jpg
    224.jpg
    343.jpg
    49.jpg
    176.jpg
    5.jpg
  potholes/
    210.jpg
    78.jpg
    89.jpg
    169.jpg
    256.jpg
    224.jpg
    49.jpg
    176.jpg
    5.jpg
    132.jpg


In [4]:
import glob

print("XML labels:", len(glob.glob(path + "/**/*.xml", recursive=True)))
print("YOLO TXT labels:", len(glob.glob(path + "/**/*.txt", recursive=True)))
print("CSV labels:", len(glob.glob(path + "/**/*.csv", recursive=True)))
print("JSON labels:", len(glob.glob(path + "/**/*.json", recursive=True)))

XML labels: 0
YOLO TXT labels: 0
CSV labels: 0
JSON labels: 0


In [5]:
import os
from collections import Counter

image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
folders = Counter()

for root, _, files in os.walk(path):
    count = sum(file.lower().endswith(image_extensions) for file in files)
    if count:
        folders[root] = count

for folder, count in folders.most_common():
    print(f"{count:5} images  |  {folder}")

  352 images  |  /root/.cache/kagglehub/datasets/atulyakumar98/pothole-detection-dataset/versions/4/normal
  329 images  |  /root/.cache/kagglehub/datasets/atulyakumar98/pothole-detection-dataset/versions/4/potholes


In [6]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

print("Classes:", train_ds.class_names)

Found 681 files belonging to 2 classes.
Using 545 files for training.
Found 681 files belonging to 2 classes.
Using 136 files for validation.
Classes: ['normal', 'potholes']


In [10]:
import tensorflow as tf

# ResNet50 pretrained on ImageNet
base_model = tf.keras.applications.ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

# Keep pretrained layers fixed during initial training
base_model.trainable = False

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.12),
    tf.keras.layers.RandomBrightness(0.15),
])

inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.resnet50.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.4)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

model.save("pothole_resnet50.keras")

Epoch 1/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 16s 522ms/step - accuracy: 0.7229 - loss: 0.5293 - val_accuracy: 0.7500 - val_loss: 0.4895
Epoch 2/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 7s 407ms/step - accuracy: 0.7853 - loss: 0.4699 - val_accuracy: 0.8456 - val_loss: 0.4211
Epoch 3/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 9s 347ms/step - accuracy: 0.8367 - loss: 0.3941 - val_accuracy: 0.8824 - val_loss: 0.3673
Epoch 4/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 10s 351ms/step - accuracy: 0.8477 - loss: 0.3720 - val_accuracy: 0.9191 - val_loss: 0.3212
Epoch 5/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 6s 338ms/step - accuracy: 0.8514 - loss: 0.3644 - val_accuracy: 0.9191 - val_loss: 0.2844
Epoch 6/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 10s 338ms/step - accuracy: 0.8862 - loss: 0.3229 - val_accuracy: 0.9191 - val_loss: 0.2545
Epoch 7/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 6s 349ms/step - accuracy: 0.9064 - loss: 0.2720 - val_accuracy: 0.9338 - val_loss: 0.2313
Epoch 8/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 7s 401ms/step - accuracy: 0.9211 - loss: 0.2418 - val_accuracy:

In [12]:
base_model.trainable = True

# Keep most pretrained layers fixed; train only the last ResNet layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

model.save("pothole_resnet50_finetuned.keras")

Epoch 1/5
18/18 ━━━━━━━━━━━━━━━━━━━━ 22s 493ms/step - accuracy: 0.8936 - loss: 0.3095 - val_accuracy: 0.9779 - val_loss: 0.1232
Epoch 2/5
18/18 ━━━━━━━━━━━━━━━━━━━━ 7s 367ms/step - accuracy: 0.9339 - loss: 0.2067 - val_accuracy: 0.9779 - val_loss: 0.1004
Epoch 3/5
18/18 ━━━━━━━━━━━━━━━━━━━━ 10s 356ms/step - accuracy: 0.9486 - loss: 0.1753 - val_accuracy: 0.9779 - val_loss: 0.0866
Epoch 4/5
18/18 ━━━━━━━━━━━━━━━━━━━━ 6s 349ms/step - accuracy: 0.9651 - loss: 0.1398 - val_accuracy: 0.9706 - val_loss: 0.0776
Epoch 5/5
18/18 ━━━━━━━━━━━━━━━━━━━━ 6s 354ms/step - accuracy: 0.9688 - loss: 0.1292 - val_accuracy: 0.9779 - val_loss: 0.0687


In [13]:
from google.colab import files
from tensorflow.keras.utils import load_img, img_to_array
import numpy as np

uploaded = files.upload()
image_name = next(iter(uploaded))

img = load_img(image_name, target_size=(224, 224))
img_array = img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)[0][0]

print("Pothole probability:", round(float(prediction) * 100, 2), "%")

if prediction >= 0.5:
    print("⚠️ Pothole detected")
else:
    print("✅ No pothole detected")

Saving images (3).jpg to images (3) (2).jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Pothole probability: 95.8 %
⚠️ Pothole detected


In [16]:
!pip -q install firebase-admin

import firebase_admin
from firebase_admin import credentials, firestore
from google.colab import files

# Upload your private Firebase service-account JSON file
uploaded = files.upload()

# Get the filename of the uploaded JSON file automatically
service_account_key = next(iter(uploaded))

cred = credentials.Certificate(service_account_key)
firebase_admin.initialize_app(cred)

db = firestore.client()
print("Firebase connected using:", service_account_key)

Saving pothole-cd57b-53fe065c5a09.json to pothole-cd57b-53fe065c5a09 (1).json
Firebase connected using: pothole-cd57b-53fe065c5a09 (1).json


In [21]:
from IPython.display import Javascript, display

display(Javascript("""
window.getLocation = function() {
  return new Promise((resolve, reject) => {
    navigator.geolocation.getCurrentPosition(
      (position) => resolve({
        latitude: position.coords.latitude,
        longitude: position.coords.longitude,
        accuracy: position.coords.accuracy
      }),
      (error) => reject(error.message),
      {
        enableHighAccuracy: true,
        timeout: 10000,
        maximumAge: 5000
      }
    );
  });
};
"""))

print("GPS function is ready.")

<IPython.core.display.Javascript object>

GPS function is ready.


In [22]:
def save_pothole(latitude, longitude, accuracy, confidence):
    doc_ref = db.collection("potholes").document()
    doc_ref.set({
        "latitude": latitude,
        "longitude": longitude,
        "accuracy": accuracy,
        "confidence": confidence,
        "timestamp": firestore.SERVER_TIMESTAMP
    })

In [24]:
from google.colab import files
from tensorflow.keras.utils import load_img, img_to_array
from google.colab.output import eval_js
import numpy as np
import urllib.request
import json

# Helper to define the JavaScript function for Geolocation in the browser
def get_gps_coordinates():
    js_code = """
    async function getLocation() {
        return new Promise((resolve, reject) => {
            navigator.geolocation.getCurrentPosition(
                (position) => {
                    resolve({
                        latitude: position.coords.latitude,
                        longitude: position.coords.longitude,
                        accuracy: position.coords.accuracy
                    });
                },
                (error) => {
                    reject(error.message);
                }
            );
        });
    }
    getLocation();
    """
    return eval_js(js_code)

# Fallback: IP-based Geolocation
def get_ip_coordinates():
    print("🛰️ Attempting IP-based geolocation fallback...")
    try:
        with urllib.request.urlopen("http://ip-api.com/json/") as response:
            data = json.loads(response.read().decode())
            if data.get("status") == "success":
                return {
                    "latitude": data.get("lat"),
                    "longitude": data.get("lon"),
                    "accuracy": 10000 # Rough estimate in meters
                }
            else:
                raise Exception("IP API responded with status: " + data.get("status", "unknown"))
    except Exception as e:
        raise Exception(f"IP geolocation failed: {e}")

# Helper to save pothole details to Firestore
def save_pothole(latitude, longitude, accuracy, confidence):
    doc_ref = db.collection("potholes").document()
    doc_ref.set({
        "latitude": latitude,
        "longitude": longitude,
        "accuracy": accuracy,
        "confidence": confidence,
        "timestamp": firestore.SERVER_TIMESTAMP
    })

uploaded = files.upload()
image_name = next(iter(uploaded))

# Prepare uploaded image for ResNet50
img = load_img(image_name, target_size=(224, 224))
img_array = img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

# Detect pothole
prediction = float(model.predict(img_array, verbose=0)[0][0])
print("Pothole probability:", round(prediction * 100, 2), "%")

if prediction >= 0.80:
    print("⚠️ Pothole detected. Requesting GPS location...")

    location = None
    try:
        # First, attempt browser Geolocation
        location = get_gps_coordinates()
    except Exception as e:
        print("ℹ️ Browser Geolocation blocked:", e)
        try:
            # Fallback to IP-based Geolocation
            location = get_ip_coordinates()
        except Exception as fallback_error:
            print("❌ Both Browser and IP Geolocation failed:", fallback_error)

    if location:
        save_pothole(
            latitude=location["latitude"],
            longitude=location["longitude"],
            accuracy=location["accuracy"],
            confidence=prediction
        )
        print("Latitude:", location["latitude"])
        print("Longitude:", location["longitude"])
        print("✅ Pothole report saved to Firebase.")
else:
    print("✅ No confident pothole detected; nothing saved.")

Saving images (3).jpg to images (3) (5).jpg
Pothole probability: 95.8 %
⚠️ Pothole detected. Requesting GPS location...
ℹ️ Browser Geolocation blocked: Geolocation has been disabled in this document by permissions policy.
🛰️ Attempting IP-based geolocation fallback...
Latitude: 45.5945
Longitude: -121.1786
✅ Pothole report saved to Firebase.
